# Agentic Evaluation: Trajectories, Tool Use, and Multi-Step Tasks

Evaluating an agent is fundamentally different from evaluating a single-turn model. An agent produces a sequence of actions over time, and the quality of any single action depends on context accumulated across the trajectory. Single-turn metrics do not compose.

## What Changes in Agentic Settings

In a single-turn LLM evaluation, the input is a prompt and the output is a response. In agentic evaluation:
- The model takes sequences of actions (tool calls, sub-agent invocations, memory reads/writes)
- Intermediate steps may be correct while the overall task fails
- Efficiency matters: an agent that takes 40 steps to complete a 5-step task is worse even if it succeeds
- Safety failures are more consequential: an agent that takes a harmful action mid-trajectory can cause irreversible damage

The evaluation unit shifts from (prompt, response) to trajectory — the full sequence of observations and actions.

## Key Metrics for Agent Evaluation

**Task completion rate**: did the agent successfully complete the end-to-end task? The primary metric.

**Step efficiency**: how many steps were taken vs. the minimum necessary? Excess steps indicate poor planning.

**Tool use precision**: did the agent call the right tools with the right arguments? Hallucinated tool calls are a common failure mode.

**Trajectory validity**: did any intermediate state violate constraints (e.g. deleted a file it shouldn't have, sent an API request to the wrong endpoint)?

**Recovery rate**: when the agent encounters an error, does it recover gracefully or spiral into repeated failure?

In [ ]:
from dataclasses import dataclass, field
from enum import Enum
from typing import Optional

class ActionType(Enum):
    TOOL_CALL = 'tool_call'
    REASONING = 'reasoning'
    FINAL_ANSWER = 'final_answer'

@dataclass
class Action:
    step: int
    action_type: ActionType
    content: str
    tool_name: Optional[str] = None
    tool_args: Optional[dict] = None
    tool_result: Optional[str] = None
    is_valid: bool = True

@dataclass
class Trajectory:
    task: str
    actions: list = field(default_factory=list)
    task_completed: bool = False
    n_steps: int = 0

    def add_action(self, action: Action):
        self.actions.append(action)
        self.n_steps += 1

def eval_trajectories(trajectories: list, available_tools: set) -> dict:
    total = len(trajectories)
    completed = sum(t.task_completed for t in trajectories)
    all_steps = [t.n_steps for t in trajectories]
    avg_steps = sum(all_steps)/len(all_steps) if all_steps else 0
    # Tool use analysis
    hallucinated_calls = 0
    total_tool_calls = 0
    for traj in trajectories:
        for action in traj.actions:
            if action.action_type == ActionType.TOOL_CALL:
                total_tool_calls += 1
                if action.tool_name not in available_tools:
                    hallucinated_calls += 1
    tool_precision = 1 - (hallucinated_calls/total_tool_calls) if total_tool_calls > 0 else 1.0
    return {
        'task_completion_rate': completed/total,
        'avg_steps': avg_steps,
        'tool_precision': tool_precision,
        'hallucinated_tool_calls': hallucinated_calls,
        'total_tool_calls': total_tool_calls,
    }

available = {'search', 'read_file', 'write_file', 'run_code'}

t1 = Trajectory(task='Find the capital of France and write it to output.txt')
t1.add_action(Action(1, ActionType.TOOL_CALL, 'searching', 'search', {'query': 'capital of France'}, 'Paris'))
t1.add_action(Action(2, ActionType.TOOL_CALL, 'writing', 'write_file', {'path': 'output.txt', 'content': 'Paris'}, 'success'))
t1.add_action(Action(3, ActionType.FINAL_ANSWER, 'Done. Wrote Paris to output.txt'))
t1.task_completed = True

t2 = Trajectory(task='Summarize document.txt')
t2.add_action(Action(1, ActionType.TOOL_CALL, 'reading', 'read_file', {'path': 'document.txt'}, 'Not found'))
t2.add_action(Action(2, ActionType.TOOL_CALL, 'using hallucinated tool', 'google_search', {'q': 'document summary'}, None))
t2.add_action(Action(3, ActionType.FINAL_ANSWER, 'Could not complete task.'))
t2.task_completed = False

results = eval_trajectories([t1, t2], available)
print('Agent evaluation results:')
for k, v in results.items():
    if isinstance(v, float):
        print(f'  {k}: {v:.2%}' if 'rate' in k or 'precision' in k else f'  {k}: {v:.1f}')
    else:
        print(f'  {k}: {v}')

## SWE-bench and Real-World Agent Benchmarks

**SWE-bench** (Jimenez et al. 2023): 2,294 real GitHub issues that require code changes to fix. The agent must read the codebase, understand the bug, and produce a patch that passes the test suite. Success rate as of 2024: ~50% for the best agents.

**WebArena**: realistic web browsing tasks (shopping, booking, information lookup) requiring multi-step navigation.

**GAIA**: 466 real-world questions requiring tool use, multi-step reasoning, and external knowledge retrieval. Specifically designed to be trivially easy for humans but hard for agents.

The common challenge across these benchmarks: agents must compose capabilities (retrieval, reasoning, tool use) over multiple steps while maintaining context.

## Trajectory-Level Safety Evaluation

Agent safety requires checking not just the final output but every intermediate action:
- Did the agent access resources it was not authorized to access?
- Did it take irreversible actions (delete, send, publish) before confirming with the user?
- Did it leak sensitive information from the context window to external tools?
- Did it follow scope constraints, or did it go beyond the task definition?

This requires logging every action with full context, which creates its own privacy and storage challenges. Production agent systems need structured audit logs that can be reviewed for safety incidents.